In [1]:
!pip install diffusers transformers accelerate torch opencv-python-headless Pillow

You should consider upgrading via the 'd:\generative deep learning\agebooth-master\agebooth-masterr\venv\scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
# %pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# %pip install --upgrade diffusers==0.21.4

import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
from diffusers.utils import load_image

d:\Generative Deep Learning\AgeBooth-master\AgeBooth-masterr\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'


In [ ]:
base_model_id = "stabilityai/stable-diffusion-xl-base-1.0"
controlnet_model_id = "diffusers/controlnet-canny-sdxl-1.0"
vae_model_id = "madebyollin/sdxl-vae-fp16-fix"

controlnet = ControlNetModel.from_pretrained(
    controlnet_model_id,
    torch_dtype=torch.float16
)

vae = AutoencoderKL.from_pretrained(
    vae_model_id,
    torch_dtype=torch.float16
)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    base_model_id,
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
)

pipe.to("cuda")

config.json: 1.31kB [00:00, 1.31MB/s]
d:\Generative Deep Learning\AgeBooth-master\AgeBooth-masterr\venv\lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
diffusion_pytorch_model.safetensors:  44%|████▎     | 2.18G/5.00G [06:2

In [ ]:
# prompt = "photorealistic portrait of an adult person, 50 years old, wise, wrinkles, **full color, vibrant, lifelike**"
# negative_prompt = "young, child, baby, smooth skin, cartoon, 3d, painting, black and white, monochrome"

prompt = "portrait of the same person aged to 80 years old, same hair color, clothes and lighting, realistic skin texture, natural aging signs, subtle wrinkles, lifelike detail, consistent identity, 8k photorealistic studio lighting, cinematic tone, ultra-detailed face."
negative_prompt = "low quality, cartoonish, deformed face, plastic skin, unrealistic lighting, wrong hair color, blurry, over-smoothed, exaggerated wrinkles, inconsistent identity, artifacts."

input_image_path = "rakshit lodu.jpg"

original_image = load_image(input_image_path).convert("RGB")
original_image = original_image.resize((1024, 1024))

image_np = np.array(original_image)

low_threshold = 100
high_threshold = 200
canny_image_np = cv2.Canny(image_np, low_threshold, high_threshold)

canny_image_np = canny_image_np[:, :, None]
canny_image_np = np.concatenate([canny_image_np, canny_image_np, canny_image_np], axis=2)

control_image = Image.fromarray(canny_image_np)

print(f"Loaded image and created Canny edge control image.")

# random seed
# seed =
generator = torch.manual_seed(42)

output_image = pipe(
    prompt,
    negative_prompt=negative_prompt,
    image=control_image,
    controlnet_conditioning_scale=0.5,
    generator=generator,
    num_inference_steps=30,
).images[0].convert("RGB")

control_image.save("control_canny_edge_image.png")
output_image.save("aged_face_output.png")
print("Successfully generated and saved 'aged_face_output.png'")
print("The edge map used as control is saved as 'control_canny_edge_image.png'")